In [49]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import json
import os

# 1. Parameters

In [50]:
REGION = "wroclaw"
DATA_PATH = f"../data/{REGION}/clean.csv"
MODEL_PATH = f"../data/{REGION}/models/"
os.makedirs(MODEL_PATH, exist_ok=True)

targets = ['NDVI', 'NDWI', 'NDMI']
look_back = 12 # one year
forecast_steps = 1 # in months

# 2. Feature engineering
## - time features

In [51]:
df = pd.read_csv(DATA_PATH)
df['Data'] = pd.to_datetime(df['Data'])
df = df.sort_values(['Sektor_ID', 'Data'])

for t in targets:
    df[f'target_{t}'] = df.groupby('Sektor_ID')[t].shift(-1)

df['day_of_year'] = df['Data'].dt.dayofyear
df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
df['month'] = df['Data'].dt.month

time_features = ['day_sin', 'day_cos', 'month']

## - lag columns

In [52]:
lag_cols = []
for col in targets:
    for i in range(1, look_back + 1):
        col_name = f'{col}_lag_{i}'
        df[col_name] = df.groupby('Sektor_ID')[col].shift(i)
        lag_cols.append(col_name)

## - delta columns

In [53]:
delta_cols = []
for col in targets:
    for i in range(1, look_back): 
        delta_name = f'{col}_delta_{i}'
        df[delta_name] = df[f'{col}_lag_{i}'] - df[f'{col}_lag_{i+1}']
        delta_cols.append(delta_name)

## - statistics

In [54]:
rolling_windows = [3, 6]
rolling_cols = []

for col in targets:
    for w in rolling_windows:
        mean_name = f'{col}_roll_mean_{w}'
        max_name = f'{col}_roll_max_{w}'
        min_name = f'{col}_roll_min_{w}'
        
        df[mean_name] = df.groupby('Sektor_ID')[col].transform(lambda x: x.rolling(window=w, min_periods=1).mean())
        df[max_name] =  df.groupby('Sektor_ID')[col].transform(lambda x: x.rolling(window=w, min_periods=1).max())
        df[min_name] =  df.groupby('Sektor_ID')[col].transform(lambda x: x.rolling(window=w, min_periods=1).min())
        
        rolling_cols.extend([mean_name, max_name, min_name])

## - spatial data

In [55]:
df_spatial = pd.read_csv(f"../data/{REGION}/spacial.csv")
df = df.merge(df_spatial, on=['Sektor_ID', 'Lat', 'Lon'], how='left')

original_cols = ['Sektor_ID', 'Data', 'Lat', 'Lon', 'NDVI', 'NDWI', 'NDMI']
spatial_cols = ['dist_to_ditch', 'manholes_300m', 'in_basin']

merged_path = f"../data/{REGION}/merged.csv"
df[original_cols + spatial_cols].to_csv(merged_path, index=False)

spatial_cols = ['dist_to_ditch', 'manholes_300m', 'in_basin']

# 3. Data preparation

In [56]:
final_features = ['NDVI', 'NDWI', 'NDMI', 'Lat', 'Lon'] + lag_cols + time_features + delta_cols + rolling_cols + spatial_cols
all_target_cols = [f'target_{t}' for t in targets]

df = df.dropna(subset=all_target_cols + lag_cols + delta_cols + rolling_cols).copy()

X = df[final_features]
y = df[all_target_cols]

X_train, X_test, y_train_all, y_test_all = train_test_split(X, y, test_size=0.2, random_state=42)


# 4. Choosing best look-back

In [57]:
import xgboost as xgb
from sklearn.metrics import r2_score

possible_lookbacks = [3, 6, 9, 12]
best_params = {}

for t in targets:
    print(f"Time window: {t}")
    best_r2 = -np.inf
    best_lb = 6
    
    for lb in possible_lookbacks:

        temp_lags = []
        df_temp = df[['Sektor_ID', 'Data', t, f'target_{t}'] + time_features + spatial_cols].copy()
        
        for i in range(1, lb + 1):
            col_name = f'lag_{i}'
            df_temp[col_name] = df_temp.groupby('Sektor_ID')[t].shift(i)
            temp_lags.append(col_name)
        
        df_temp = df_temp.dropna().copy()
        
        if len(df_temp) < 20: continue
            
        X_tmp = df_temp[temp_lags + time_features + spatial_cols]
        y_tmp = df_temp[f'target_{t}']
        
        X_tr, X_te, y_tr, y_te = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=42)
        
        m_tmp = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=6,
            random_state=42
        )

        m_tmp.fit(X_tr, y_tr)
        
        current_r2 = r2_score(y_te, m_tmp.predict(X_te))
        print(f"   - Test lb={lb}: R2 = {current_r2:.4f}")
        
        if current_r2 > best_r2:
            best_r2 = current_r2
            best_lb = lb
            
    print(f"Wybrano lb={best_lb} dla {t} (R2: {best_r2:.4f})\n")
    best_params[t] = best_lb

Time window: NDVI
   - Test lb=3: R2 = 0.7464
   - Test lb=6: R2 = 0.7745
   - Test lb=9: R2 = 0.8312
   - Test lb=12: R2 = 0.8467
Wybrano lb=12 dla NDVI (R2: 0.8467)

Time window: NDWI
   - Test lb=3: R2 = 0.6621
   - Test lb=6: R2 = 0.7006
   - Test lb=9: R2 = 0.7813
   - Test lb=12: R2 = 0.7567
Wybrano lb=9 dla NDWI (R2: 0.7813)

Time window: NDMI
   - Test lb=3: R2 = 0.5546
   - Test lb=6: R2 = 0.5873
   - Test lb=9: R2 = 0.7631
   - Test lb=12: R2 = 0.7644
Wybrano lb=12 dla NDMI (R2: 0.7644)



# 5. Model training

In [58]:
models = {}

for t in targets:
    
    print(f"Training final model for: {t} (using lb={best_params[t]})")
    
    current_lag_features = [f'{col}_lag_{i}' for col in targets for i in range(1, best_params[t] + 1)]
    current_delta_features = [f'{col}_delta_{i}' for col in targets for i in range(1, best_params[t])]
    
    current_features = (
        ['NDVI', 'NDWI', 'NDMI', 'Lat', 'Lon'] + 
        current_lag_features + 
        current_delta_features + 
        rolling_cols + 
        spatial_cols + 
        time_features
    )
    
    X_train_t = X_train[current_features]
    X_test_t = X_test[current_features]
    y_train = y_train_all[f'target_{t}']
    y_test = y_test_all[f'target_{t}']
    
    m = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
    
    m.fit(X_train_t, y_train, eval_set=[(X_test_t, y_test)], verbose=False)
    
    print(f"- final R2 Score: {r2_score(y_test, m.predict(X_test_t)):.4f}")
    models[t] = m
    models[t].feature_names = current_features

Training final model for: NDVI (using lb=12)
- final R2 Score: 0.8467
Training final model for: NDWI (using lb=9)
- final R2 Score: 0.7770
Training final model for: NDMI (using lb=12)
- final R2 Score: 0.8233


# 6. Forecast

In [60]:
current_batch = df.groupby('Sektor_ID').tail(1).copy()
recursive_results = []

for step in range(1, forecast_steps + 1):

    preds = {}
    for t in targets:

        model_features = models[t].feature_names_in_ 
        X_curr_for_target = current_batch[model_features]
        preds[t] = models[t].predict(X_curr_for_target)
    
    temp_res = current_batch[['Sektor_ID', 'Lat', 'Lon'] + spatial_cols].copy()
    for t in targets:
        temp_res[f'{t}'] = preds[t]
    
    forecast_date = current_batch['Data'].max() + pd.DateOffset(months=step)
    temp_res['Forecast_Date'] = forecast_date
    temp_res['Step'] = step
    recursive_results.append(temp_res)

    for i in range(look_back, 1, -1):
        for t in targets:
            current_batch[f'{t}_lag_{i}'] = current_batch[f'{t}_lag_{i-1}']
    
    for t in targets:
        current_batch[f'{t}_lag_1'] = current_batch[t]
        current_batch[t] = preds[t]
        
    current_batch['day_of_year'] = forecast_date.dayofyear
    current_batch['day_sin'] = np.sin(2 * np.pi * current_batch['day_of_year'] / 365.25)
    current_batch['day_cos'] = np.cos(2 * np.pi * current_batch['day_of_year'] / 365.25)
    current_batch['month'] = forecast_date.month

    for t in targets:
        for i in range(1, look_back):
            current_batch[f'{t}_delta_{i}'] = current_batch[f'{t}_lag_{i}'] - current_batch[f'{t}_lag_{i+1}']

forecast_df = pd.concat(recursive_results)

OUTPUT_PATH = f"../data/{REGION}/forecast_1m.csv"
forecast_df.to_csv(OUTPUT_PATH, index=False)